# 02 — Feature Engineering Analysis

This notebook visualizes the custom features extracted from the LTMM accelerometer and posturography signals.

We extracted 129 features including:
- Time-Domain (mean, variance, RMS, max/min, range, skew, kurtosis, IQR)
- Jerk (RMS, mean)
- Frequency-Domain (dominant frequency, spectral entropy, band power [low/mid/high])
- Postural Sway (sway area, path length, mean velocity from AP/ML acceleration trajectories)

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from posturisk.preprocess import DEFAULT_PROCESSED_DIR

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
%matplotlib inline

## 1. Load Feature Matrix

In [ ]:
df = pd.read_csv(DEFAULT_PROCESSED_DIR / 'features.csv')
print(f"Loaded {df.shape[0]} subjects with {df.shape[1]} features")

# Separate clinical and signal features for visualization
clinical_cols = ['age', 'gender', 'year_fall', 'six_month_fall', 'abc_pct', 'pase', 'mmse', 'moca', 'fab', 'tmta', 'tmtb', 'tug', 'fsst', 'berg', 'dgi']
signal_cols = [c for c in df.columns if c not in clinical_cols and c not in ['subject_id', 'is_faller', 'sample_rate', 'n_samples', 'duration_s']]

print(f"Signal features count: {len(signal_cols)}")

## 2. Visualize Postural Sway Features (Fallers vs Non-Fallers)

In [ ]:
sway_cols = ['sway_path_length', 'sway_area', 'sway_mean_velocity']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, sway_cols):
    sns.boxplot(x='is_faller', y=col, data=df, ax=ax, palette=['#2ecc71', '#e74c3c'], width=0.4)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xticklabels(['Non-Faller', 'Faller'])
    ax.set_xlabel('')

plt.suptitle('Postural Sway Dynamics', y=1.05)
plt.tight_layout()
plt.show()

## 3. Visualize Frequency Band Powers


In [ ]:
band_cols = ['v_acc_power_low', 'v_acc_power_mid', 'v_acc_power_high']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, band_cols):
    sns.boxplot(x='is_faller', y=col, data=df, ax=ax, palette=['#2ecc71', '#e74c3c'], width=0.4)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xticklabels(['Non-Faller', 'Faller'])
    ax.set_xlabel('')
    
plt.suptitle('Vertical Acceleration Frequency Bands\nLow (0.1-3Hz), Mid (3-10Hz), High (10-20Hz)', y=1.08)
plt.tight_layout()
plt.show()

## 4. Signal Acceleration and Jerk RMS

In [ ]:
dynamics_cols = ['v_acc_rms', 'v_acc_jerk_rms', 'acc_magnitude_rms']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, dynamics_cols):
    sns.boxplot(x='is_faller', y=col, data=df, ax=ax, palette=['#2ecc71', '#e74c3c'], width=0.4)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xticklabels(['Non-Faller', 'Faller'])
    ax.set_xlabel('')
    
plt.suptitle('Signal Magnitude & Jerk (Smoothness)', y=1.05)
plt.tight_layout()
plt.show()

## 5. Feature Correlation Heatmap (Selected Features)

In [ ]:
# Selecting a diverse mix of features to see their correlation
focus_features = ['is_faller', 'age', 'tug', 'berg', 
                  'sway_area', 'sway_mean_velocity',
                  'v_acc_rms', 'v_acc_jerk_rms',
                  'v_acc_power_high']

corr = df[focus_features].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, vmin=-1, vmax=1, fmt=".2f", square=True,
            cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap: Clinical vs Advanced Signal Features')
plt.tight_layout()
plt.show()